# Stage 5: Text Preprocessing
This notebook cleans raw comments and transcript lines using standard Indonesian NLP rules (normalization, URL/emoji removal, tokenization, stopword removal, and Sastrawi stemming).

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

import sys, os
from pathlib import Path
cwd = Path(os.getcwd()).resolve()
base_dir = cwd if (cwd / 'config').exists() else (cwd.parent if (cwd.parent / 'config').exists() else cwd)
if str(base_dir) not in sys.path: sys.path.insert(0, str(base_dir))

from config import settings
from src.preprocessing.text_preprocessor import preprocess_comment

## 1. Load Parquet Corpus

In [ ]:
df = pd.read_parquet(settings.PARQUET_PATH)
print(f"Loaded {len(df)} comments.")
print("Sample raw comment:", df.iloc[0]['text'])

## 2. Apply Clean & Preprocess
*( Pruning spam comments before running preprocessor )*

In [ ]:
# Filter out spam
clean_df = df[df["is_spam"] == 0].copy()
print(f"Pruned spam. Clean comments count: {len(clean_df)}")

processed_comments = []
for _, row in tqdm(clean_df.iterrows(), total=len(clean_df)):
    res = preprocess_comment(row["text"])
    processed_comments.append({
        "comment_id": row["comment_id"],
        "video_id": row["video_id"],
        "parent_id": row["parent_id"],
        "cleaned_text": res["cleaned"],
        "stemmed_text": res["stemmed"],
        "tokens": ",".join(res["tokens"])
    })

processed_df = pd.DataFrame(processed_comments)
processed_df.to_parquet(settings.PROCESSED_DATA_DIR / "comments_processed.parquet", index=False)
print("Processed comments saved to data/processed/comments_processed.parquet")

## 3. Review Preprocessed Sample

In [ ]:
print(processed_df.head(2).to_dict(orient="records"))